# LightGBM Final Model Training and Validation

This notebook uses the best hyperparameters identified from the Optuna tuning notebook to train the final LightGBM model.  

Steps included:  
1. Load and prepare the dataset for modeling.  
2. Split data into train, validation, and test sets using time-based thresholds.  
3. Define features, target, and categorical columns for LightGBM.  
4. Train LightGBM with early stopping on the validation set.  
5. Predict on validation set and compute RMSE using only rows where actual prices exist.

In [1]:
# Data manipulation and modeling imports
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

# Custom utilities for data handling
from utils.data_splitter import DataSplitter
from utils.wrangle_model_data import wrangle_ml

In [2]:
# Load and prepare data
df = pd.read_csv('data/commodity_prices.csv')
df = wrangle_ml(df)

# Define train, validation, and test date ranges
date_dict = {
    'train_start': "2023-06-01", 'train_end': "2025-06-30",
    'valid_start': "2025-07-01", 'valid_end': "2025-07-31",
    'test_start': "2025-08-01", 'test_end': "2025-08-18"
}

# Minimum rows required per split
thresholds = {'train': 250, 'valid': 10, 'test': 5}

# Create train, validation, and test splits
splitter = DataSplitter(df, date_dict, thresholds)
train_df, valid_df, test_df = splitter.run()

## LightGBM Model Training and Validation

In this section, we:

1. Define features and target for the model.  
2. Prepare training and validation feature matrices and target vectors.  
3. Specify categorical columns for LightGBM and convert them to `category` dtype.  
4. Load the best hyperparameters obtained from the Optuna tuning notebook.  
5. Create LightGBM dataset objects and train the model with early stopping.  
6. Predict on the validation set and filter only rows with actual prices for evaluation.

In [3]:
# Define features and target
features = [col for col in train_df.columns if col not in ['log_Modal_Price', 'log_Modal_Price_filled', 'Arrival_Date']]
target_col = "log_Modal_Price_filled"

# Prepare feature matrices and target vectors
X_train, y_train = train_df[features].copy(), train_df[target_col].copy()
X_val, y_val = valid_df[features].copy(), valid_df[target_col].copy()

# Specify categorical columns for LightGBM
categorical_cols = ['Product_Type', 'Commodity', 'Variety_Type', 'Market', 'Season', 
                    'Market_Season', 'Variety_Type', 'Product_Month']

# Convert categorical columns to 'category' dtype
for c in categorical_cols:
    X_train[c] = X_train[c].astype('category')
    X_val[c] = X_val[c].astype('category')

# Best hyperparameters from Optuna tuning
best_params = {
    'learning_rate': 0.0439,
    'num_leaves': 89,
    'max_depth': 4,
    'min_child_samples': 64,
    'subsample': 0.6932,
    'colsample_bytree': 0.6071,
    'reg_lambda': 6.13e-06
}

# Create LightGBM dataset objects
lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_cols)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train, categorical_feature=categorical_cols)

# Train the LightGBM model with early stopping
model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=0)  # suppress verbose logs
    ]
)

# Predict on validation set
y_pred = model.predict(X_val, num_iteration=model.best_iteration)

# Only evaluate rows where original prices exist
mask_val = valid_df['log_Modal_Price'].notna()
y_val_actual = y_val[mask_val]
y_pred_actual = y_pred[mask_val]

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4664
[LightGBM] [Info] Number of data points in the train set: 204620, number of used features: 27
[LightGBM] [Info] Start training from score 8.364827
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


## Validation Performance

We compute the Root Mean Squared Error (RMSE) on the validation set using only rows where actual prices exist.  
This metric provides an estimate of the model's predictive accuracy on unseen data.

In [4]:
# Compute RMSE on actual validation values
rmse = root_mean_squared_error(y_val_actual, y_pred_actual)
print("Validation RMSE:", rmse)

Validation RMSE: 0.11012495356852899


### Validation RMSE

The final LightGBM model achieved a **validation RMSE of 0.1101**, indicating good predictive accuracy on the validation set.